# Spark Environment
This notebook includes examples of how to connect and configure Spark to run in a local environment and to manage the dependencies and options required to read/write data to an S3 comptabile object storage.

### Environment Notes
* By convention credentials and secrets are injected into the runtime as environment variables.
  * BASH environment variables are usually named in all caps.
  * Scala variables are generally lowercase.
* AWS/S3 variables are prefixed using `OBJECTS_*`

## Configure Runtime
Jupyter can be configured to work with [BeakerX](http://beakerx.com/), a set of kernels and extensions which provide JVM support, Spark cluster support, and interactive plots/tables/forms.

* `%%classpath`: One of BeakerX's "magics" provides support for dynamically loading dependencies through the use of Maven.

In [ ]:
%%classpath add mvn
org.apache.hadoop hadoop-common 3.1.2
org.apache.spark spark-sql_2.11 2.4.5
org.apache.spark spark-kubernetes_2.11 2.4.5
com.amazonaws aws-java-sdk-s3 1.11.744

Packages:

* Spark SQL: Spark's structured data interface
* Spark Kubernetes: Set of tools and utilities that allows Spark to run workers on Kubernetes.
* Amazon AWS Java SDK for Amazon S3

In [ ]:
%classpath add jar /opt/spark/jars/joda-time-2.9.9.jar
%classpath add jar /opt/spark/jars/httpclient-4.5.3.jar
%classpath add jar /opt/spark/jars/aws-java-sdk-s3-1.11.534.jar
%classpath add jar /opt/spark/jars/aws-java-sdk-kms-1.11.534.jar
%classpath add jar /opt/spark/jars/aws-java-sdk-dynamodb-1.11.534.jar
%classpath add jar /opt/spark/jars/aws-java-sdk-core-1.11.534.jar
%classpath add jar /opt/spark/jars/aws-java-sdk-1.11.534.jar
%classpath add jar /opt/spark/jars/hadoop-aws-3.1.2.jar
%classpath add jar /opt/spark/jars/slf4j-api-1.7.29.jar
%classpath add jar /opt/spark/jars/slf4j-log4j12-1.7.29.jar

_JARS included in the commands above are needed to allow Spark to read and write data from MinIO. They are included as part of the Spark dependencies in the container._

***

## Input/Output With Spark
Spark is able to read data from a large number of sources and write data to a large number of sinks.

* Many of the storage classes are provided by Hadoop, showing an example of where Spark builds on top of the development done by other projects.

### Configure Spark Context and Context
_See [Spark documentation](https://spark.apache.org/docs/latest/configuration.html) for a reference of configuration values and options._

In [ ]:
// Retrieve S3 credentials from environment for Spark environment config
val OBJECT_STORAGE_URL = sys.env.getOrElse("OBJECTS_ENDPOINT", "")
val OBJECT_STORAGE_ACCESSID = sys.env.getOrElse("OBJECTS_ACCESSID", "")
val OBJECT_STORAGE_SECRET = sys.env.getOrElse("OBJECTS_SECRET", "")
val OBJECTS_PLAYGROUND_BUCKET = "playground"
val OBJECTS_PREFIX = sys.env.getOrElse("HOSTNAME", "")

println(s"Object storage URL: $OBJECT_STORAGE_URL")

In [ ]:
// Import SparkConf and SparkContext
import org.apache.spark.{SparkConf,SparkContext}
import org.apache.spark.sql.SparkSession

// Create configuration 
val conf = new SparkConf()
    .set("spark.driver.memory", "2G")
    .set("spark.executor.memory", "1G")
    .set("spark.driver.maxResultSize", "1G")
    .set("spark.driver.host", sys.env.getOrElse("HOSTNAME", ""))

// Storage configuration
conf.set("spark.hadoop.fs.s3a.endpoint", OBJECT_STORAGE_URL)
    .set("spark.hadoop.fs.s3a.access.key", OBJECT_STORAGE_ACCESSID)
    .set("spark.hadoop.fs.s3a.secret.key", OBJECT_STORAGE_SECRET)
    .set("spark.hadoop.fs.s3a.fast.upload", "true")
    .set("spark.hadoop.fs.s3a.path.style.access", "true")
    .set("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")

// Set application master and hostname. The local[2] means run two threads "minimal"
// parallelism which allows for bugs to be located.
conf.setMaster("local[2]")
    .setAppName("iotest")

// Create the Spark Context and Session
val sc = new SparkContext(conf)
val spark = SparkSession.builder()
    .appName("iotest").getOrCreate()

In [ ]:
// Import Spark implicits, which allow the use of $-notation
import spark.implicits._

### Check that Spark Session is Active

In [ ]:
// Create a distributed data to test the session
val t = sc.parallelize(1 to 1000 by 2)

// Pull data from workers to foreground thread
t.take(10).foreach(println)

### Write Data to MinIO
Spark is able to read and write data to object storage by specifying the file path details in the URL. This has the general form:

`s3a://{{ bucket }}/{{ object-key-path }}`. Example: `s3a://oak-tree.tech/hello-test.txt`

In [ ]:
// Create S3 URL to write test data to
val s3_testurl_csv = s"s3a://$OBJECTS_PLAYGROUND_BUCKET/$OBJECTS_PREFIX/colors-test.csv"
println(s3_testurl_csv)

In [ ]:
// Create data frame from sequence/rdd. Apply column names
val cnames = Seq("name", "color")
val df1 = spark.createDataFrame(
        sc.parallelize(
            Seq(
                ("Mario", "Red"), 
                ("Luigi", "Green"), 
                ("Princess", "Pink"))))
    .toDF(cnames: _*)

df1.show()

In [ ]:
// Write data to object storage (CSV)
df1.write.mode("overwrite").option("header", true).csv(s3_testurl_csv)

#### Write Alternative File Formats to MinIO
Input/output in Spark aims to "just work." The type of storage is specified by an output path, with the type of output specified by an output type and options.

In [ ]:
// Create Parquet files in MinIO (CSV)
val s3_testurl_parquet = s"s3a://$OBJECTS_PLAYGROUND_BUCKET/$OBJECTS_PREFIX/colors-test.parquet"
df1.write.mode("overwrite").parquet(s3_testurl_parquet)

#### Read Data from MinIO
CSV

In [ ]:
// Read data as structured CSV
val df1scv = spark.read.format("csv")
    .option("inferSchema", "true")
    .option("header", "true")
    .option("sep", ",")
    .load(s3_testurl_csv)

df1scv.show()

In [ ]:
df1scv.printSchema()

Parquet

In [ ]:
val df4 = spark.read.format("parquet").load(s3_testurl_parquet)

df4.printSchema()

### Load Remote Data/Infer Schema
Spark is able to attempt to infer a schema when loading string data (CSV/TSV):

* Controlled using the `inferSchema` option

_Before running the commands in this section, the code in "Working with Remote Data Sources" needs to have been executed._

In [ ]:
val framingham_s3key = s"s3a://$OBJECTS_PLAYGROUND_BUCKET/$OBJECTS_PREFIX/framingham.csv"
println(framingham_s3key)

In [ ]:
val df3 = spark.read.format("csv")
    .option("inferSchema", "true")
    .option("header", "true")
    .option("sep", ",")
    .load(framingham_s3key)

df3.printSchema()

In [ ]:
df3.select("male", "age", "education", "BMI", 
    "diaBP", "sysBP", "TenYearCHD").show()